# Part 1: Clustering
## CSL7110 - Machine Learning with Big Data | Assignment 4
**Name:** Aniket Srivastava | **Roll No:** M25DE1051

This notebook implements:
- **Farthest First Traversal** for k-center clustering
- **k-Means++** probabilistic seeding
- **k-Means Objective** function
- Coreset experiment: kcenter coreset + kmeans++

**Dataset:** UCI Spambase (4601 points, 58 features, last column = label)

In [ ]:
import numpy as np
import time
import random

random.seed(42)
np.random.seed(42)

## Task 1: readVectorsSeq

In [ ]:
# Task 1: Read feature vectors
def readVectorsSeq(filename):
    """Read comma-separated vectors; last column is class label and is dropped."""
    vectors = []
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = list(map(float, line.split(',')))
            vectors.append(np.array(vals[:-1], dtype=np.float64))
    return vectors

P = readVectorsSeq('../data/Q1/spambase.data')
print(f"Loaded {len(P)} points with {len(P[0])} dimensions")

Loaded 4601 points with 57 dimensions


## Task 2: kcenter - Farthest First Traversal

In [ ]:
# Task 2: k-Center Clustering - Farthest First Traversal O(|P|*k)
def kcenter(P, k):
    """Farthest First Traversal: O(|P|*k). Returns k center vectors."""
    n = len(P)
    first_idx = random.randint(0, n - 1)
    centers = [P[first_idx]]
    min_dists = np.array([np.sum((p - centers[0])**2) for p in P], dtype=np.float64)
    for _ in range(k - 1):
        farthest_idx = int(np.argmax(min_dists))
        new_c = P[farthest_idx]
        centers.append(new_c)
        new_dists = np.array([np.sum((p - new_c)**2) for p in P], dtype=np.float64)
        min_dists = np.minimum(min_dists, new_dists)
    return centers

print("kcenter function defined")

kcenter function defined


## Task 3: kmeansPP - k-Means++ Seeding

In [ ]:
# Task 3: k-Means++ Probabilistic Seeding O(|P|*k)
def kmeansPP(P, k):
    """k-Means++ D^2 weighted sampling. Returns k center vectors."""
    n = len(P)
    first_idx = random.randint(0, n - 1)
    centers = [P[first_idx]]
    dists = np.array([np.sum((p - centers[0])**2) for p in P], dtype=np.float64)
    for _ in range(k - 1):
        total = dists.sum()
        if total == 0:
            idx = random.randint(0, n - 1)
        else:
            probs = dists / total
            idx = int(np.random.choice(n, p=probs))
        new_c = P[idx]
        centers.append(new_c)
        new_dists = np.array([np.sum((p - new_c)**2) for p in P], dtype=np.float64)
        dists = np.minimum(dists, new_dists)
    return centers

print("kmeansPP function defined")

kmeansPP function defined


## Task 4: kmeansObj - k-Means Objective

In [ ]:
# Task 4: k-Means Objective: avg squared distance to nearest center
def kmeansObj(P, C):
    """Returns sum of min squared distances to centers / |P|."""
    C_arr = np.array(C)
    total = 0.0
    for p in P:
        sq_dists = np.sum((C_arr - p)**2, axis=1)
        total += sq_dists.min()
    return total / len(P)

print("kmeansObj function defined")

kmeansObj function defined


## Step 1: Run kcenter(P, k)

In [ ]:
k  = 10
k1 = 100

random.seed(42); np.random.seed(42)

print(f"[Step 1] kcenter(P, k={k})")
t0 = time.time()
C_kc = kcenter(P, k)
elapsed_kc = time.time() - t0
obj1 = kmeansObj(P, C_kc)
print(f"  Running time        : {elapsed_kc:.4f} s")
print(f"  kmeansObj (kcenter) : {obj1:.4f}")

[Step 1] kcenter(P, k=10)
  Running time        : 0.1924 s
  kmeansObj (kcenter) : 86240.4746


## Step 2: Run kmeansPP(P, k) then kmeansObj

In [ ]:
random.seed(42); np.random.seed(42)

print(f"[Step 2] kmeansPP(P, k={k}) -> kmeansObj")
t0 = time.time()
C_kpp = kmeansPP(P, k)
elapsed_kpp = time.time() - t0
obj2 = kmeansObj(P, C_kpp)
print(f"  Running time        : {elapsed_kpp:.4f} s")
print(f"  kmeansObj(P, C)     : {obj2:.4f}")

[Step 2] kmeansPP(P, k=10) -> kmeansObj
  Running time        : 0.1717 s
  kmeansObj(P, C)     : 44366.9662


## Step 3: Coreset - kcenter(P, k1) -> kmeansPP(X, k) -> kmeansObj

In [ ]:
random.seed(42); np.random.seed(42)

print(f"[Step 3] kcenter(P, k1={k1}) -> kmeansPP(X, k={k}) -> kmeansObj")
t0 = time.time()
X = kcenter(P, k1)
elapsed_x = time.time() - t0
print(f"  kcenter(P, k1) time : {elapsed_x:.4f} s  |  |X| = {len(X)}")

t0 = time.time()
C_cs = kmeansPP(X, k)
elapsed_cs = time.time() - t0
obj3 = kmeansObj(P, C_cs)
print(f"  kmeansPP(X, k) time : {elapsed_cs:.4f} s")
print(f"  kmeansObj(P, C)     : {obj3:.4f}")

print()
print("Summary")
print(f"  Step 1 kcenter obj        : {obj1:.4f}")
print(f"  Step 2 kmeansPP obj       : {obj2:.4f}")
print(f"  Step 3 coreset(k1)+kpp obj: {obj3:.4f}")
print()
print("kmeansPP on full data gives lowest objective (best centers).")
print("Coreset is a 2-approx representative sample; larger k1 -> better coreset.")

[Step 3] kcenter(P, k1=100) -> kmeansPP(X, k=10) -> kmeansObj
  kcenter(P, k1) time : 1.5598 s  |  |X| = 100
  kmeansPP(X, k) time : 0.0053 s
  kmeansObj(P, C)     : 333643.9539

Summary
  Step 1 kcenter obj        : 86240.4746
  Step 2 kmeansPP obj       : 44366.9662
  Step 3 coreset(k1)+kpp obj: 333643.9539

kmeansPP on full data gives lowest objective (best centers).
Coreset is a 2-approx representative sample; larger k1 -> better coreset.
